# PRISM — the five missing PAIP slides

Separate notebook because PRISM needs its own environment. Run this in the PRISM env, not the one used for
`Feature_Extraction_Missing_PAIP_fivecrop.ipynb`.

Slides:

`training_data_19_nonMSIH` · `training_data_30_MSIH` · `training_data_41_nonMSIH` · `training_data_42_MSIH` · `training_data_46_nonMSIH`

**Run steps 1 and 2 of `Feature_Extraction_Missing_PAIP_fivecrop.ipynb` first.** They write the metadata and
add these slides to `final_merged_without_white.csv`. Without that, the filter below finds nothing and this
notebook does nothing. The first cell checks for it and tells you if it's missing.

Two stages, same as `Feature_Extraction_Prism2_fivecrop.ipynb`:

1. **Patches** — Virchow **V1** (not V2 — PRISM was trained on V1), five crops averaged into one 2560-d vector
   per patch → `Features\Prism_patches\<slide>\<patch>.pt`
2. **Slides** — stack a slide's patch vectors, run PRISM's Perceiver once → `(1, 1280)` →
   `Features\Prism_slides\<slide>.pt`

In [1]:
import os
import glob

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

PAIP_ROOT      = r"D:\Aamir Gulzar\KSA_project2\paip_data"
PATCH_DATA_DIR = os.path.join(PAIP_ROOT, "patch_data")
NON_WHITE_CSV  = os.path.join(PAIP_ROOT, "final_merged_without_white.csv")
PATCH_FEAT_ROOT  = os.path.join(PAIP_ROOT, "Features", "Prism_patches")
SLIDE_EMBED_ROOT = os.path.join(PAIP_ROOT, "Features", "Prism_slides")

SLIDES = [
    "training_data_19_nonMSIH",
    "training_data_30_MSIH",
    "training_data_41_nonMSIH",
    "training_data_42_MSIH",
    "training_data_46_nonMSIH",
]

CROP_SIZE = 256

# check the white filter has been run for these slides
valid_names = set(pd.read_csv(NON_WHITE_CSV)["patch_name"].astype(str))

missing = []
for slide in SLIDES:
    n = sum(1 for name in valid_names if name.startswith(slide + "_"))
    print(f"{slide:<26} {n:>6,} non-white patches")
    if n == 0:
        missing.append(slide)

if missing:
    raise ValueError(
        f"No rows in final_merged_without_white.csv for: {', '.join(missing)}\n"
        "Run steps 1 and 2 of Feature_Extraction_Missing_PAIP_fivecrop.ipynb first."
    )

training_data_19_nonMSIH      898 non-white patches
training_data_30_MSIH       1,466 non-white patches
training_data_41_nonMSIH      170 non-white patches
training_data_42_MSIH       1,116 non-white patches
training_data_46_nonMSIH      569 non-white patches


In [2]:
import timm
from timm.data import resolve_data_config
from timm.layers import SwiGLUPacked
from torchvision import transforms
from transformers import AutoModel
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not set - PRISM and Virchow are gated repos")
login(token=hf_token)
print("logged in to hugging face")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in to hugging face


## The model

Two pieces: Virchow V1 encodes patches, PRISM's Perceiver turns a slide's worth of patch vectors into one
slide embedding.

Note the crops are **averaged** into a single 2560-d vector per patch — that's what PRISM expects in the tile
sequence, so unlike the other models there's no crop dimension in the saved file.

In [3]:
class Prism:
    def __init__(self, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        print("loading PRISM...")
        self.prism = AutoModel.from_pretrained(
            "paige-ai/Prism", trust_remote_code=True
        ).to(self.device).eval()

        # Virchow V1, not V2 - PRISM's Perceiver was trained on V1 embeddings
        print("loading Virchow V1...")
        self.tile_encoder = timm.create_model(
            "hf-hub:paige-ai/Virchow", pretrained=True,
            mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU, num_classes=0,
        ).to(self.device).eval()

        cfg = resolve_data_config(self.tile_encoder.pretrained_cfg, model=self.tile_encoder)
        self.eval_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=cfg["mean"], std=cfg["std"]),
        ])
        self.fivecrop = transforms.Compose([
            transforms.FiveCrop(CROP_SIZE),
            transforms.Lambda(lambda crops: torch.stack([self.eval_transform(c) for c in crops])),
        ])
        print("ready")

    @torch.inference_mode()
    def encode_patch(self, path):
        """One patch -> one 2560-d vector (the five crops get averaged)."""
        img = Image.open(path).convert("RGB")
        crops = self.fivecrop(img).to(self.device)

        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            out = self.tile_encoder.forward_features(crops)   # (5, tokens, 1280)
            cls_token  = out[:, 0]                            # (5, 1280)
            patch_mean = out[:, 5:].mean(dim=1)               # (5, 1280), skip registers
            combined   = torch.cat([cls_token, patch_mean], dim=-1)   # (5, 2560)

        return combined.mean(dim=0).cpu().float()             # (2560,)

    @torch.inference_mode()
    def encode_slide(self, patch_embeddings):
        """All of a slide's patch vectors (N, 2560) -> one (1, 1280) embedding."""
        tiles = patch_embeddings.unsqueeze(0).to(self.device)   # (1, N, 2560)
        return self.prism.slide_representations(tiles)["image_embedding"].cpu()

## Stage 1 — encode the patches

One `.pt` per patch. Already-encoded patches are skipped, so you can stop and restart this.

In [4]:
prism = Prism()

loading PRISM...
loading Virchow V1...
ready


In [5]:
for slide in SLIDES:
    out_dir = os.path.join(PATCH_FEAT_ROOT, slide)
    os.makedirs(out_dir, exist_ok=True)

    patches = glob.glob(os.path.join(PATCH_DATA_DIR, slide, "*.png"))
    patches = [p for p in patches if os.path.basename(p) in valid_names]

    done = {f[:-3] for f in os.listdir(out_dir) if f.endswith(".pt")}
    todo = [p for p in patches if os.path.splitext(os.path.basename(p))[0] not in done]

    if not todo:
        print(f"{slide}: all {len(patches):,} patches already done")
        continue

    print(f"{slide}: encoding {len(todo):,} of {len(patches):,} patches")
    for path in tqdm(todo, desc=f"  {slide}", unit="patch", leave=False):
        stem = os.path.splitext(os.path.basename(path))[0]
        embedding = prism.encode_patch(path)
        torch.save(embedding, os.path.join(out_dir, stem + ".pt"))

    print(f"  done, shape {tuple(embedding.shape)}")

training_data_19_nonMSIH: encoding 898 of 898 patches


  training_data_19_nonMSIH:   0%|          | 0/898 [00:00<?, ?patch/s]c:\Users\datainsight\anaconda3\envs\prism_env\lib\site-packages\timm\models\vision_transformer.py:91: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  x = F.scaled_dot_product_attention(


  done, shape (2560,)
training_data_30_MSIH: encoding 1,466 of 1,466 patches


  done, shape (2560,)
training_data_41_nonMSIH: encoding 170 of 170 patches


  done, shape (2560,)
training_data_42_MSIH: encoding 1,116 of 1,116 patches


  done, shape (2560,)
training_data_46_nonMSIH: encoding 569 of 569 patches


  done, shape (2560,)


## Stage 2 — slide embeddings

Stack each slide's patch vectors and run the Perceiver **once** per slide.

In [6]:
os.makedirs(SLIDE_EMBED_ROOT, exist_ok=True)

for slide in SLIDES:
    save_path = os.path.join(SLIDE_EMBED_ROOT, slide + ".pt")
    if os.path.exists(save_path):
        print(f"{slide}: slide embedding already exists")
        continue

    pt_files = sorted(glob.glob(os.path.join(PATCH_FEAT_ROOT, slide, "*.pt")))
    if not pt_files:
        print(f"{slide}: no patch features, run stage 1 first")
        continue

    patch_embeddings = torch.stack([torch.load(p, map_location="cpu") for p in pt_files])
    print(f"{slide}: {len(pt_files):,} patches -> {tuple(patch_embeddings.shape)}")

    slide_embedding = prism.encode_slide(patch_embeddings)
    torch.save(slide_embedding, save_path)
    print(f"  saved {tuple(slide_embedding.shape)} -> {save_path}")

training_data_19_nonMSIH: 898 patches -> (898, 2560)
  saved (1, 1280) -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Prism_slides\training_data_19_nonMSIH.pt
training_data_30_MSIH: 1,466 patches -> (1466, 2560)
  saved (1, 1280) -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Prism_slides\training_data_30_MSIH.pt
training_data_41_nonMSIH: 170 patches -> (170, 2560)
  saved (1, 1280) -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Prism_slides\training_data_41_nonMSIH.pt
training_data_42_MSIH: 1,116 patches -> (1116, 2560)
  saved (1, 1280) -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Prism_slides\training_data_42_MSIH.pt
training_data_46_nonMSIH: 569 patches -> (569, 2560)
  saved (1, 1280) -> D:\Aamir Gulzar\KSA_project2\paip_data\Features\Prism_slides\training_data_46_nonMSIH.pt


## Check

Compare the five against a slide that was already finished.

In [7]:
# reference: one of the 73 that were already done
old_slide = next(d for d in sorted(os.listdir(PATCH_FEAT_ROOT)) if d not in SLIDES)
old_file  = next(f for f in os.listdir(os.path.join(PATCH_FEAT_ROOT, old_slide)) if f.endswith(".pt"))
ref_patch = tuple(torch.load(os.path.join(PATCH_FEAT_ROOT, old_slide, old_file),
                             map_location="cpu").shape)
ref_slide = tuple(torch.load(os.path.join(SLIDE_EMBED_ROOT, old_slide + ".pt"),
                             map_location="cpu").shape)

print(f"reference ({old_slide}): patch {ref_patch}, slide {ref_slide}")
print(f"patch folders: {len(os.listdir(PATCH_FEAT_ROOT))}")
print(f"slide files  : {len([f for f in os.listdir(SLIDE_EMBED_ROOT) if f.endswith('.pt')])}\n")

for slide in SLIDES:
    pts = glob.glob(os.path.join(PATCH_FEAT_ROOT, slide, "*.pt"))
    expected = sum(1 for name in valid_names if name.startswith(slide + "_"))

    embed_path = os.path.join(SLIDE_EMBED_ROOT, slide + ".pt")
    if not pts or not os.path.exists(embed_path):
        print(f"{slide:<26} INCOMPLETE ({len(pts):,} patches)")
        continue

    patch_shape = tuple(torch.load(pts[0], map_location="cpu").shape)
    embedding   = torch.load(embed_path, map_location="cpu")
    ok = (patch_shape == ref_patch and tuple(embedding.shape) == ref_slide
          and len(pts) == expected and torch.isfinite(embedding).all())

    print(f"{slide:<26} {len(pts):>6,} / {expected:<6,} patches {patch_shape} "
          f"slide {tuple(embedding.shape)} {'ok' if ok else 'CHECK'}")

reference (training_data_01_MSIH): patch (2560,), slide (1, 1280)
patch folders: 78
slide files  : 78

training_data_19_nonMSIH      898 / 898    patches (2560,) slide (1, 1280) ok
training_data_30_MSIH       1,466 / 1,466  patches (2560,) slide (1, 1280) ok
training_data_41_nonMSIH      170 / 170    patches (2560,) slide (1, 1280) ok
training_data_42_MSIH       1,116 / 1,116  patches (2560,) slide (1, 1280) ok
training_data_46_nonMSIH      569 / 569    patches (2560,) slide (1, 1280) ok
